# Ateliê Generativo — Treinamento LoRA (Google Colab / GPU T4)

Notebook-guia da **Etapa 2**. Treina o adaptador LoRA de pixel art 16 bits.

**Antes de começar:** Runtime → Alterar tipo de runtime → **GPU (T4)**.

**1 - Infraestrutura, Otimização e Armazenamento Persistente**

In [ ]:
# Célula 1: Montagem do Drive e Instalação do Ecossistema
from google.colab import drive
drive.mount('/content/drive')

import os
# Cria o diretório de destino no seu Google Drive
drive_path = "/content/drive/MyDrive/Atelie_Generativo_LoRA"
os.makedirs(drive_path, exist_ok=True)
print(f"📁 Pasta de salvamento dos modelos criada em: {drive_path}")

# Instalação das bibliotecas com versões otimizadas para GPU T4
!pip install -q diffusers transformers accelerate peft bitsandbytes datasets

# Otimização crítica de atenção
!pip install -q xformers==0.0.25 --no-deps

# Clonagem do repositório da Hugging Face para acesso ao script de treinamento
!git clone https://github.com/huggingface/diffusers.git

**2 - Ingestão de Dados e Mapeamento de Caminhos (Atualizado)**

In [ ]:
# Célula 2: Download do Dataset e Construção do metadata.jsonl
import json

# Clona o repositório da equipe
!git clone https://github.com/Thiagofariasl/atelie-generativo-estilo-pixel-art.git

# Definição dos caminhos base
dataset_dir = "/content/atelie-generativo-estilo-pixel-art/dataset"
txt_file = os.path.join(dataset_dir, "legendas.txt")
jsonl_file = os.path.join(dataset_dir, "metadata.jsonl")

# Lógica de conversão considerando a arquitetura do GitHub
# Transforma: "img_01.jpg: legenda" -> {"file_name": "images/img_01.jpg", "text": "legenda"}
with open(txt_file, 'r', encoding='utf-8') as f_in, open(jsonl_file, 'w', encoding='utf-8') as f_out:
    for linha in f_in:
        if ':' in linha:
            arquivo, legenda = linha.split(':', 1)

            # Adicionando o prefixo da subpasta para garantir que o Diffusers encontre a imagem
            caminho_relativo = f"images/{arquivo.strip()}"

            json_obj = {"file_name": caminho_relativo, "text": legenda.strip()}
            f_out.write(json.dumps(json_obj) + '\n')

print("✅ Dataset preparado! O arquivo metadata.jsonl mapeia as imagens na subpasta 'images/'.")

**3 - Configuração do Accelerate**

In [ ]:
# Célula 3: Configuração do hardware
from accelerate.utils import write_basic_config
write_basic_config()
print("🚀 Accelerate configurado para maximizar a eficiência da GPU T4!")

**4 - O Treinamento Matemático (As Duas Configurações Exigidas)**

> Execute a Célula 4A, aguarde terminar (pode levar 20-40 min dependendo do número de épocas/imagens).
> Quando concluir, execute a 4B.

In [ ]:
# Célula 4A: TESTE A (Rank 8) - Salvando direto no Google Drive
# Rank 8: Rápido, aprende características globais (cores, blocos do pixel art).
# Pode ter dificuldade com detalhes complexos.
!accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_name="/content/atelie-generativo-estilo-pixel-art/dataset" \
  --dataloader_num_workers=8 \
  --resolution=512 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-04 \
  --max_grad_norm=1 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=0 \
  --output_dir="/content/drive/MyDrive/Atelie_Generativo_LoRA/PixelArt_Rank8" \
  --checkpointing_steps=500 \
  --seed=42 \
  --rank=8 \
  --mixed_precision="fp16" \
  --enable_xformers_memory_efficient_attention

In [ ]:
# Célula 4B: TESTE B (Rank 32) - Salvando direto no Google Drive
# Rank 32: Demora um pouco mais, captura detalhes muito mais finos do traço e do estilo,
# mas corre um leve risco de decorar (overfitting) se o dataset for pequeno.
!accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_name="/content/atelie-generativo-estilo-pixel-art/dataset" \
  --dataloader_num_workers=8 \
  --resolution=512 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-04 \
  --max_grad_norm=1 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=0 \
  --output_dir="/content/drive/MyDrive/Atelie_Generativo_LoRA/PixelArt_Rank32" \
  --checkpointing_steps=500 \
  --seed=42 \
  --rank=32 \
  --mixed_precision="fp16" \
  --enable_xformers_memory_efficient_attention

**5 - Inferência e Validação Visual**
> Após o treino, os arquivos *.safetensors* dos seus LoRAs estarão a salvo no seu Google Drive. Use esta célula para carregar o modelo treinado e testar se ele aprendeu o estilo!

In [ ]:
# Célula 5: Teste de Geração (Inferência)
import torch
from diffusers import StableDiffusionPipeline

# 1. Carrega o modelo base
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipe.to("cuda")

# 2. Injeta os pesos do seu LoRA treinado (Vamos testar o Rank 32)
caminho_do_lora = "/content/drive/MyDrive/Atelie_Generativo_LoRA/PixelArt_Rank32"
pipe.load_lora_weights(caminho_do_lora)

# 3. Geração! Não esqueça de usar a sua Trigger Word ("estilo_pixel_art,")
prompt = "estilo_pixel_art, cena de um leão caminhando em uma floresta digital, cores vibrantes em pixel art 16 bits."

print("🎨 Gerando imagem...")
imagem = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]

# Exibe a imagem no Colab
imagem.show()